# Введение и подготовка

## Что такое регулярные выражения?

Вы наверняка умеете пользоваться поиском по странице (Ctrl+F или Cmd+F). Это отличный инструмент, когда вы точно знаете, что ищете. Например, слово <b>"Привет"</b>.

Но что делать, если задача сложнее? Представьте, что у вас есть огромный документ на 1000 страниц, и вам нужно найти:
- Все <b>номера телефонов</b>
- Все <b>email-адреса</b> пользователей
- Все <b>даты</b>, упомянутые в тексте

Обычный поиск здесь бессилен, потому что вы ищете не конкретный текст, а <b>структуру</b> или <b>шаблон</b>. Здесь на сцену выходят они - <b>Регулярные выражения</b>.

### Что это такое?

<b>Регулярные выражения</b> (<b>Regular Expressions</b>, сокращенно <b>RegEx</b> или <b>RegExp</b>) - это специальный язык для описания шаблонов поиска в тексте. Если обычный поиск ищет буквальное совпадение (символ в символ), то регулярные выражения ищут совпадение по правилу.

### Где это используется?

Регулярные выражения - это навык-суперсила. Они используются везде, от простых текстовых редакторов до сложных систем безопасности. Вот 3 самые частые задачи, которые вы будете решать:
1. <b>Валидация данных (Проверка).</b> 
Пользователь ввел email при регистрации. Нам нужно проверить, действительно ли это почта, а не просто набор букв.
    - <b>Без RegEx</b>: Написать 20 строк кода с кучей условий if и циклов
    - <b>С RegEx</b>: Одна строка кода с шаблоном <b>^\S+@\S+\.\S+$</b>
2. <b>Поиск и извлечение (Парсинг).</b>
У вас есть лог-файл сервера с миллионом строк. Нужно вытащить только IP-адреса тех, кто пытался взломать систему. RegEx позволяет <b>"выудить"</b> из текста нужные кусочки данных.

3. <b>Замена и очистка.</b>
Удалить все HTML-теги из текста, оставив только слова? Заменить формат даты с 2023-10-05 на 05.10.2023 во всем документе разом? Для RegEx это дело пары секунд.


### Python и RegEx

Python имеет встроенную поддержку регулярных выражений через стандартный модуль re. Вам не нужно ничего устанавливать дополнительно.

## Подготовка в Python: Сырые строки (r"...")

Работа с регулярными выражениями в Python с импорта модуля:

In [1]:
import re

Модуль <b>re (Regular Expressions)</b> входит в стандартную библиотеку, устанавливать его не нужно. Но прежде чем писать код, нам нужно разобрать <b>самое важное правило</b> синтаксиса, из-за которого новички часто путаются и тратят часы на поиск ошибок.

### Проблема двойного дна

Главная сложность в том, что ваше регулярное выражение проходит через <b>два фильтра</b>, прежде чем начать поиск:
1. <b>Интерпретатор Python</b>. Он читает вашу строку кода.
2. <b>Движок RegEx</b>. Он получает то, что осталось после Python, и использует это как инструкцию для поиска.

#### Конфликт спецсимволов

Символ \ (обратный слэш) используется как спецсимвол и в Python, и в RegEx.
- <b>В Python</b>: \n - перенос строки, \t - табуляция.
- <b>В RegEx</b>: \d - цифра, \w - буква, \\. - экранирование точки.

Это создает конфликт. Допустим, мы хотим найти в тексте два символа: \n (слэш и букву n). Если написать pattern = '\n', Python превратит это в невидимый символ переноса строки. Движок RegEx получит "пустоту" (Enter) вместо шаблона. Чтобы передать в RegEx именно два символа (\ и n), в обычном Python нужно экранировать слэш: pattern = '\\n'.

А если мы хотим найти <b>сам символ слэша</b> \?
- RegEx требует \\\ (чтобы экранировать спецсимвол).
- Чтобы передать \\\ через Python, каждый слэш нужно экранировать снова.
- Итог: pattern = '\\\\'. Это называют "Чумой обратных слэшей"

### Спасение: Raw strings (Сырые строки)

Чтобы не сходить с ума, в Python есть сырые строки. Они обозначаются префиксом r перед кавычками: r'...'

|Код|Что делает Python|Что в итоге получает RegEx|
|-|-|-|
|'\n'|Превращает в символ переноса строки|(Enter)|
|r'\n'|Ничего, оставляет символы \ и n|\n (шаблон поиска переноса)|
|'\\\d'|Экранирует слэш, оставляет \ и d|\d (шаблон цифры)|
|r'\d'|Ничего, оставляет \ и d|\d (шаблон цифры)|

### Важный нюанс. Слэш в квадрате.

Студенты часто спрашивают: "Если я использую r'...', почему я не могу написать r'C:\\' для поиска пути?" 

Здесь вступает в игру <b>Второй фильтр (Движок RegEx)</b>. Да, префикс r говорит Питону не трогать слэши. Но когда строка попадает в <b>RegEx</b>, слэш там - это <b>всё ещё спецсимвол</b>! 

Если вы отправите в RegEx строку <b>C:\\</b>, движок RegEx посмотрит на слэш в конце и будет ждать продолжения команды (например, \d). А продолжения нет. Это приведет к ошибке.

<b>Правило поиска слэша</b>:
Чтобы найти <b>один</b> реальный слэш в тексте, движку RegEx нужно дать инструкцию: <b>два</b> слэша \\\ (первый экранирует второй).

Поэтому даже в сырых строках:
- Хотим найти путь: C:\Windows
- Пишем паттерн: r'C:\\Windows'

#### Почему нельзя писать r'\'? (SyntaxError)

Есть техническое ограничение Python: сырая строка не может заканчиваться одним нечетным слэшем. Код <b>pattern = r'abc\'</b> выдаст ошибку. Причина: Python считает, что этот слэш "экранирует" закрывающую кавычку, и строка не заканчивается. Если вам очень нужно, чтобы строка заканчивалась слэшем, используйте конкатенацию: <b>r'C:\' + '\\'</b> или просто пишите два слэша <b>r'C:\\'</b>, если это для RegEx.

## Первый инструмент: re.match()

Настало время написать первый код. Самая базовая функция модуля - это re.match().

### Синтаксис.

Функция принимает два обязательных аргумента:

In [2]:
# re.match(pattern, string)

1. <b>pattern</b> - шаблон регулярного выражения (то, что мы ищем)
2. <b>string</b> - строка, в которой мы ищем

### Важная особенность: Строгий привратник.

Главное, что нужно запоминить про <b>re.match()</b>: <b><i>Эта функция ищет совпадение ТОЛЬКО в начале строки</i></b>. Если искомый текст находится хотя бы на один символ дальше от начала, match скажет, что ничего не нашел. Он не сканирует строку целиком, он проверяет только её старт.

### Примеры

Давайте посмотрим, как это работает на практике.

<b>Пример 1: Успешное совпадение</b>.

Мы ищем слово "Hello" в начале строки.

In [ ]:
import re

text = 'Hello world'
pattern = r'Hello'

result = re.match(pattern, text)
print(result)

<re.Match object; span=(0, 5), match='Hello'>


Функция вернула <b>Match object</b>. Это означает "Успех".

<b>Пример 2: Неудача (текст в середине)</b>

Теперь попробуем найти "Hello", но поставим его в середину предложения.

In [4]:
import re

text = 'Oh, Hello world'
pattern = r'Hello'

result = re.match(pattern, text)
print(result)

None


Функция вернула <b>None</b> (пустоту). Несмотря на то, что "Hello" есть в строке, <b>re.match</b> его не увидел, так как строка начинается не с него.

### Как использовать в коде?

Так как <b>re.match</b> возвращает либо объект (который считается как <b>True</b>), либо <b>None</b> (который считается как <b>False</b>), его очень удобно использовать в условиях <b>if</b>.

In [5]:
import re

text = input('Введите лог: ')
# Проверяем, начинается ли строка со слова Error
if re.match(r'Error', text):
    print('Тревога! Найдена ошибка.')
else:
    print('Все спокойно.')

Все спокойно.


### Резюме

- re.match(pattern, string) - проверяет, подходит ли начало строки под шаблон.
- Если подходит - возвращает объект <re.Match>.
- Если не подходит - возвращает None.

P.S. Если вам нужно найти текст где угодно внутри строки, а не только в начале, для этого существует другая функция, о которой мы поговорим чуть позже.

## Задачи.

### Задача 1: Первый контакт (Import и Match)

<b>Условие</b>:

Напишите программу, которая считывает строку и проверяет, начинается ли она со слова RegEx (с большой буквы).
1. Импортируйте модуль re.
2. Используйте re.match с сырой строкой r"RegEx".
3. Если совпадение есть — выведите Match.
4. Если совпадения нет — выведите No match.

In [6]:
import re

pattern = r'RegEx'
text = input()

print('Match' if re.match(pattern, text) else 'No match')

Match


### Задача 2: Опасный префикс (HTTPS)

<b>Условие</b>:

Нужно проверить протокол сайта. Если введенная ссылка начинается с https, сайт считается защищенным. Если нет — незащищенным.

Напишите регулярное выражение для поиска строки https.

Secure выведите, если ссылка начинается с https.

В противном случае Insecure.

In [8]:
import re

pattern = r'https'
text = input()

print(re.match(pattern, text) and 'Secure' or 'Insecure')

Insecure


### Задача 3: Работаем со слэшами (Windows path)

<b>Условие</b>:

Самый сложный момент для новичка — экранирование.

Напишите программу, которая проверяет, начинается ли введенный путь с диска C:\ (C, двоеточие, обратный слэш). Если пусть начинается с дикса C, то напиши Disk C. Если нет, то выведи Other.

<b>Важно</b>: Чтобы найти один обратный слэш \\, в сырой строке r"..." нужно написать два слэша \\\\.

In [ ]:
import re

pattern = r'C:\\'
text = input()

print(re.match(pattern, text) and 'Disk C' or 'Other')

Other


### Задача 4: Команда для бота

<b>Условие</b>:

Мы пишем Telegram-бота. Команды должны начинаться со слэша /. Проверьте, начинается ли введенное сообщение с команды /start. Если введенное сообщение начинается с команды /start выводим - Command recognized

Если же нет, то - <b>Text message</b>

In [13]:
import re

pattern = r'/start'
text = input()

print(re.match(pattern, text) and 'Command recognized' or 'Text message')

Text message


### Задача 5: Скобки в тексте

<b>Условие</b>:

Нам нужно найти строку, которая начинается с открывающей квадратной скобки [.
Квадратная скобка — это спецсимвол в RegEx (вы узнаете об этом в следующем модуле), поэтому её нужно <b>экранировать</b> с помощью \\.

Создайте шаблон, который ищет буквальное совпадение символа [.

Если подходит под условие, выводим - Bracket found.
В противном случае - No bracket at start

In [15]:
import re

pattern = r'\['
text = input()

print(re.match(pattern, text) and 'Bracket found' or 'No bracket at start')

No bracket at start


## Результат поиска: Пан или пропал

Когда мы запускаем функции re.match() или re.search(), происходит одно из двух событий. Третьего не дано.

### 1. Неудача: None

Если шаблон не подходит под строку, функция возвращает специальный тип данных - None (Ничего).

In [16]:
import re

# Ищем цифры, а в строке только буквы
result = re.match(r'\d+', 'abc')
print(result)
# Вывод: None

None


Это не строка "None", это пустота. Это означает False в логическом контексте.

### 2. Успех: Match object

Если шаблон найден, функция возвращает объект совпадения (Match object). Многие новички ожидают увидеть просто найденную строку (например, "123"), но Python дает нам нечто большее - целый контейнер с информацией.

In [18]:
# Ищем буквы
result = re.match(r'[a-z]+', 'abc')
print(result)

<re.Match object; span=(0, 3), match='abc'>


Посмотрите на вывод. Это "коробка", внутри которой лежит:
1. <b>match='abc'</b>: Сама найденная строка.
2. <b>span=(0, 3)</b>: Координаты (индексы), где эта строка находится в тексте (от 0 до 3).

### Осторожно: AttributeError

Самая популярная ошибка при работе с re выглядит так: AttributeError: 'NoneType' object has no attribute 'group'

<b>Почему она возникает?</b>

Вы пытаетесь достать данные из результата поиска, не проверив, нашелся ли он вообще.

In [ ]:
# ОШИБОЧНЫЙ КОД
result = re.match(r'X', 'abc') # Вернет None
print(result.group()) # Ошибка! У пустоты нет методов.

### Битва методов: re.match() против re.search()

В предыдущем уроке мы запускали re.match(). В следующем мы будем использовать re.search(). Новички часто путают их или считают синонимами. Но это <b>разные инструменты,</b> и выбор неправильного метода - причина №1, почему скрипт "ничего не находит".

1. <b>re.match() - Строгий швейцар</b>

Этот метод проверяет наличие шаблона строго в начале строки. Если искомый текст сдвинут хотя бы один символ вправо - match вернет None (Неудачу).
- <b>Логика</b>: "Начинается ли строка с этого шаблона?"

In [20]:
import re

text = "Я люблю Python"

# Пытаемся найти "Python" с помощью match
# Шаблон "Python" есть в тексте, но он НЕ в начале.
result = re.match(r"Python", text)

print(result)

None


<b>Почему?</b> Потому что строка начинается с буквы "Я". re.match посмотрел на первый символ, увидел несовпадение и мгновенно прекратил работу.

2. <b>re.search() - Настойчивый детектив</b>

Этот метод сканирует <b>всю строкуф</b> целиком, символ за символом. Он ищет <b>первое</b> совпадение, где бы оно ни находилось: в начале, в середине или в конце.
- <b>Логика:</b> "Есть ли этот шаблон хоть где-нибудь в тексте?"

In [21]:
import re

text = "Я люблю Python"

# Пытаемся найти "Python" с помощью search
result = re.search(r"Python", text)

print(result)

<re.Match object; span=(8, 14), match='Python'>


## Анатомия совпадения: Разбираем Match object.

Итак, наша проверка if result: прошла успешно. У нас в руках есть объект Match. Но сам по себе объект - это просто техническая обертка. Нам нужно достать из него полезные данные.

Для этого у объекта 4 главных метода. Рассмотрим их на примере. Представим, что мы ищем слово <b>"Code"</b> в строке.

In [22]:
import re

text = "I love Code and Coffee"
match = re.search(r"Code", text)

print(match)

<re.Match object; span=(7, 11), match='Code'>


1. <b>.group() - Самое главное</b>

Метод .group() возвращает <b>саму подстроку</b>, которая была найдена.

In [23]:
print(match.group())

Code


Это то, что вы будете использовать в 90% случаев.
- На заметку: Можно написать <b>match.group(0)</b> - это то же самое. (О цифрах в скобках мы поговорим в модуле про Группировку).

2. <b>Координаты: .start() и .end()</b>

Иногда нам важно знать не только что нашлось, но и где это лежит (например, чтобы подсветить текст в редакторе).
- <b>.start()</b> - возвращает индекс <b>начала</b> совпадения (включительно).
- <b>.end()</b> - возвращает индекс <b>конца</b> совпадения (не включительно).

In [24]:
print(match.start())
print(match.end())

7
11


<b>Важный нюанс Python:</b> Индекс .end() указывает на символ, который идет сразу после найденного слова. Это сделано для удобства срезов (slicing).

3. <b>.span() - Всё и сразу</b>

Метод .span() возвращает кортеж (tuple) из двух чисел: (start, end).

In [25]:
print(match.span())

(7, 11)


<b>Магия срезов</b>

Благодаря тому, как работают .start() и .end(), в Python работает следующая математика:

In [26]:
s, e = match.span()
print(text[s:e])

Code


## Задачи.

### Задача 1. Что именно мы нашли? (.group)

<b>Условие</b>:

Вам нужно найти слово Python в начале введенной строки.

1. Используйте re.match с шаблоном r"Python".
2. Если совпадение есть, выведите найденную строку, используя метод .group().
3. Если совпадения нет, выведите строку Not found.

(Мы пока ищем только фиксированное слово, так как спецсимволы изучаем в следующем модуле).



In [31]:
import re

pattern = r'Python'
text = input()
match = re.match(pattern, text)

print(match and match.group() or 'Not found')

Not found


### Задача 2. Где начало? (.start)

<b>Условие</b>:

Мы используем функцию <b>re.search</b>, чтобы найти слово error в любом месте строки (функцию search мы упоминали, она возвращает такой же Match объект).

Ваша задача — найти позицию, с которой начинается слово error.

1. Если слово найдено, выведите строку в формате: Start index: X, где X — результат метода .start().
2. Если слово не найдено, выведите No error.

In [ ]:
import re

pattern = r'error'
text = input()
match = re.search(pattern, text)

print(match and f'Start index: {match.start()}' or 'No error')

Start index: 0
<re.Match object; span=(0, 5), match='error'>


### Задача 3. Координаты (.span)

<b>Условие:</b>

Функция <b>re.match</b> возможно нашла слово <b>Code</b> в начале строки. Вам нужно вывести кортеж с координатами начала и конца этого совпадения.
1. Шаблон для поиска: <b>r"Code"</b>.
2. Если совпадение есть, выведите результат метода <b>.span()</b>.
3. Если совпадения нет, выведите <b>None</b>.

In [ ]:
import re

pattern = r'Code'
text = input()
match = re.match(pattern, text)

print(match and match.span() or None)

### Задача 4. Проверка на безопасность (AttributeError)

<b>Условие</b>:

Это задача-ловушка. Многие новички пишут print(re.match(...).group()) сразу, не проверяя результат.
Напишите программу, которая ищет слово Key в начале строки.
- Если нашли — выведите Found.
- Если не нашли — выведите Missing.

<b>Важно</b>: Не пытайтесь вызывать методы у результата, если там None, иначе решение упадет с ошибкой AttributeError на скрытых тестах.

In [ ]:
import re

pattern = r'Key'
text = input()

print(re.match(pattern, text) and 'Found' or 'Missing')

### Задача 5. Срез строки (Slice)

<b>Условие</b>:

Помните, что метод <b>.end()</b> возвращает индекс символа после совпадения? Это идеально подходит для срезов.
<b>Задача</b>: найти слово Log в начале строки и вывести всё, что идет после него.

1. Найдите Log с помощью re.match.
2. Если нашли, используйте match.end(), чтобы сделать срез text[...] от конца найденного слова до конца строки. Выведите этот срез.
3. Если не нашли, выведите пустую строку (просто print()).

In [ ]:
import re

pattern = r'Log'
text = input()
match = re.match(pattern, text)

print(match and text[match.end():] or '')